# Dataset inspection — streaming summary stats over `DvcDataset`

Walk every store under `TRANSDUCED_ROOT`, run `inspect_dataset` with a tqdm progress bar, and roll the per-array stats into a DataFrame for triage. No voxel reads happen until the inspector runs.

**`RECOMPUTE` flag.** The inspector is expensive on full-size volumes (≈15 min / store at `(960, 1280, 1280)` float32). Set `RECOMPUTE = True` to re-run `inspect_dataset` and overwrite cached artifacts under `dataset-insights/`. With `RECOMPUTE = False` (default) the notebook loads `DatasetStats` from `dataset_stats.pkl` and rebuilds the DataFrames from it. Discovery, opening, and the convention check always run live since they are cheap.

See `docs/plans/dataset-inspect.md` for the design contract and `mamba_dvc/io/inspect.py` for the implementation.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from mamba_dvc.io.dataset import DvcDataset
from mamba_dvc.io.inspect import (
    DatasetStats,
    FlowLayoutError,
    inspect_dataset,
    print_reporter,
    tqdm_reporter,
    verify_flow_convention,
)

# Flip to True to re-run inspect_dataset and refresh the cached artifacts
# under INSIGHTS_DIR. With RECOMPUTE=False the notebook loads the pickled
# DatasetStats and rebuilds the DataFrames from it; the convention check
# still runs live because it only touches a small centered subblock.
RECOMPUTE = False

INSIGHTS_DIR = Path('dataset-insights')
INSIGHTS_DIR.mkdir(exist_ok=True)
STATS_PKL = INSIGHTS_DIR / 'dataset_stats.pkl'
REF_PKL = INSIGHTS_DIR / 'reference_stats.pkl'
MASK_PKL = INSIGHTS_DIR / 'mask_stats.pkl'
DEFORM_PKL = INSIGHTS_DIR / 'deformation_stats.pkl'
CONVENTION_PKL = INSIGHTS_DIR / 'convention_checks.pkl'

## 1. Discover stores

Adjust `TRANSDUCED_ROOT` to point at your local data root. Mirrors the convention in `notebooks/datainspection.ipynb`.

In [ ]:
TRANSDUCED_ROOT = Path(r'D:\jannik\synchrotron-data\transduced')

store_paths = sorted(TRANSDUCED_ROOT.glob('*.zarr'))
for p in store_paths:
    print(p.name)
print(f'\n{len(store_paths)} store(s) found')

## 2. Open lazily

`strict=False` keeps this triage-friendly: per-entry issues land in `dataset.broken_entries` and are forwarded by the inspector into `DatasetStats.skipped` rather than raising.

In [ ]:
datasets: dict[str, DvcDataset] = {}
for p in store_paths:
    try:
        datasets[p.stem] = DvcDataset.open(p, strict=False)
    except Exception as exc:
        print(f'[skip] {p.name}: {exc}')

for name, ds in datasets.items():
    print(
        f'{name}: shape={ds.volume_shape} '
        f'masks={ds.list_masks()} '
        f'real={len(ds.list_real())} synthetic={len(ds.list_synthetic())} '
        f'broken={len(ds.broken_entries)}'
    )

## 3. Run the inspector (or load the cache)

Default median strategy is `"histogram"` — bounded memory, ≤0.006% relative resolution at 16384 bins. The cached run below uses `median="skip"` for speed; switch to `median="exact"` if you need bit-exact medians on a `dry_shape` subblock.

When `RECOMPUTE=True` (or the pickle is missing) the inspector runs per store with a tqdm bar and serializes the full `dict[str, DatasetStats]` to `dataset_stats.pkl`. Otherwise the cached dict is unpickled. The flow-layout check still runs up-front during recomputation; a malformed store fails fast with `FlowLayoutError`.

In [ ]:
if RECOMPUTE or not STATS_PKL.exists():
    stats: dict[str, DatasetStats] = {}
    for name, ds in datasets.items():
        print(f'\n=== {name} ===')
        try:
            stats[name] = inspect_dataset(ds, progress=tqdm_reporter(), median='skip')
        except FlowLayoutError as exc:
            print(f'[layout] {name}: {exc}')
        except Exception as exc:
            print(f'[error] {name}: {type(exc).__name__}: {exc}')
    with STATS_PKL.open('wb') as f:
        pickle.dump(stats, f)
    print(f'\n[cache] wrote {STATS_PKL} ({len(stats)} datasets)')
else:
    with STATS_PKL.open('rb') as f:
        stats = pickle.load(f)
    print(
        f'[cache] loaded {STATS_PKL} ({len(stats)} datasets) '
        f'— set RECOMPUTE=True to refresh'
    )

## 4. Roll into DataFrames

Three views: per-dataset reference, per-mask, per-deformation.

In [ ]:
ref_rows = [
    {
        'dataset': name,
        'shape': s.reference.shape,
        'dtype': s.reference.dtype,
        'mean': s.reference.mean,
        'min': s.reference.min,
        'max': s.reference.max,
        'median': s.reference.median,
        'method': s.reference.median_method,
        'elapsed_s': s.elapsed_seconds,
    }
    for name, s in stats.items()
]
ref_df = pd.DataFrame(ref_rows)
ref_df.to_pickle(REF_PKL)
ref_df

In [ ]:
mask_rows = [
    {
        'dataset': dname,
        'mask': mname,
        'foreground': ms.foreground_count,
        'background': ms.background_count,
        'fg_fraction': ms.foreground_fraction,
    }
    for dname, s in stats.items()
    for mname, ms in s.masks.items()
]
mask_df = pd.DataFrame(mask_rows)
mask_df.to_pickle(MASK_PKL)
mask_df

In [ ]:
deform_rows = []
for dname, s in stats.items():
    for name, ds_stats in s.deformations.items():
        row = {
            'dataset': dname,
            'name': name,
            'kind': ds_stats.kind,
            'image_mean': ds_stats.image.mean,
            'image_min': ds_stats.image.min,
            'image_max': ds_stats.image.max,
            'image_median': ds_stats.image.median,
        }
        if ds_stats.flow is not None:
            row.update(
                flow_mean_mag=ds_stats.flow.mean_magnitude,
                flow_max_mag=ds_stats.flow.max_magnitude,
                flow_median_mag=ds_stats.flow.median_magnitude,
                flow_mean_dz=ds_stats.flow.per_axis_mean[0],
                flow_mean_dy=ds_stats.flow.per_axis_mean[1],
                flow_mean_dx=ds_stats.flow.per_axis_mean[2],
                flow_max_abs_dz=ds_stats.flow.per_axis_max_abs[0],
                flow_max_abs_dy=ds_stats.flow.per_axis_max_abs[1],
                flow_max_abs_dx=ds_stats.flow.per_axis_max_abs[2],
                flow_axis_order=ds_stats.flow.axis_order,
            )
        deform_rows.append(row)
deform_df = pd.DataFrame(deform_rows)
deform_df.to_pickle(DEFORM_PKL)
deform_df

## 5. Convention check

Empirical verification of which sign convention each synthetic entry was generated with. `verify_flow_convention` warps the reference under the stored flow with both `"pull_back"` and `"push_forward"` and picks the convention whose residual against the stored deformed image is smaller — the smaller residual identifies the convention the producer actually used, and `agrees` cross-checks that against the value declared in the manifest / profile (the `declared` column).

Rows where `agrees=False` are flagged in red. Those stores need a manifest fix before `correlate()` will return signed-correct displacements, since `GroundTruthField.from_zarr` would apply the wrong sign flip otherwise.

`ratio = max(residual) / min(residual)` is a confidence indicator. Values near `1.0` mean the field is too small to discriminate (e.g. a near-zero displacement); large values mean the smaller residual is decisively smaller. The default `dry_shape=(32, 64, 64)` keeps the test cheap (sub-second per entry); expand it if a near-1.0 ratio leaves the result inconclusive.

In [ ]:
if RECOMPUTE or not CONVENTION_PKL.exists():
    convention_rows = []
    for dname, ds in datasets.items():
        print(f'\n=== {dname} ===')
        try:
            checks = verify_flow_convention(ds)
        except FlowLayoutError as exc:
            print(f'[layout] {dname}: {exc}')
            continue
        except Exception as exc:
            print(f'[error] {dname}: {type(exc).__name__}: {exc}')
            continue
        for entry_name, check in checks.items():
            convention_rows.append(
                {
                    'dataset': dname,
                    'entry': entry_name,
                    'declared': check.declared,
                    'empirical': check.empirical,
                    'agrees': check.agrees,
                    'residual_pull_back': check.residual_pull_back,
                    'residual_push_forward': check.residual_push_forward,
                    'ratio': check.ratio,
                    'dry_shape': check.dry_shape,
                }
            )
    convention_df = pd.DataFrame(convention_rows)
    convention_df.to_pickle(CONVENTION_PKL)
    print(f'\n[cache] wrote {CONVENTION_PKL} ({len(convention_df)} entries)')
else:
    convention_df = pd.read_pickle(CONVENTION_PKL)
    print(
        f'[cache] loaded {CONVENTION_PKL} ({len(convention_df)} entries) '
        f'— set RECOMPUTE=True to refresh'
    )


def _flag_disagreement(row: pd.Series) -> list[str]:
    style = 'background-color: #ffe0e0' if not row['agrees'] else ''
    return [style] * len(row)


(
    convention_df.style.apply(_flag_disagreement, axis=1)
    if not convention_df.empty
    else convention_df
)

## 6. Skipped entries

Entries the dataset bound as `BrokenEntry` are forwarded with their reason. The inspector never tries to read voxels for these.

In [ ]:
skipped_rows = [
    {'dataset': dname, 'name': name, 'reason': reason}
    for dname, s in stats.items()
    for name, reason in s.skipped.items()
]
skipped_df = pd.DataFrame(skipped_rows) if skipped_rows else pd.DataFrame(
    columns=['dataset', 'name', 'reason']
)
skipped_df

## 7. Fast-triage variant

When you only need fg fractions and intensity ranges (no medians), `median="skip"` is roughly half the disk reads. Runs live against a single store — independent of the cached `stats` dict above.

In [ ]:
if datasets:
    sample_name, sample_ds = next(iter(datasets.items()))
    print(f'fast triage on {sample_name}')
    fast = inspect_dataset(sample_ds, median='skip', progress=print_reporter)
    print(f'\nelapsed: {fast.elapsed_seconds:.2f}s')

## 8. Subset / single-deformation drill-down

Pass `include=` to focus on a specific deformation, e.g. when investigating a flagged synthetic case. Runs live, like the fast-triage variant.

In [ ]:
if datasets:
    sample_name, sample_ds = next(iter(datasets.items()))
    target = next(iter(sample_ds.list_synthetic()), None)
    if target is not None:
        focused = inspect_dataset(
            sample_ds,
            include=[target],
            masks=[],
            progress=print_reporter,
        )
        d = focused.deformations[target]
        print(f'\n{sample_name}::{target} ({d.kind})')
        print(f'  image: mean={d.image.mean:.3f} median={d.image.median:.3f}')
        if d.flow is not None:
            print(
                f'  flow:  mean|u|={d.flow.mean_magnitude:.3f} '
                f'max|u|={d.flow.max_magnitude:.3f} '
                f'per-axis-mean={d.flow.per_axis_mean}'
            )